In [0]:
/*
================================================================================
VISTA: supermercadoetl_prod.gold.catalogo_productos
CAPA: Gold (Analítica - Cálculo de Recetas)
ENTORNO: PRODUCCIÓN
AUTOR: Sistema ETL Supermercados
FECHA CREACIÓN: 2026-04-17
ÚLTIMA MODIFICACIÓN: 2026-04-17
================================================================================

PROPÓSITO:
Vista materializada que normaliza productos del catálogo Silver para cálculo 
de precio unitario según tipo de medida (peso/volumen/unidad). Permite calcular
costos de recetas de manera automatizada y escalable.

DEPENDENCIAS:
- Tabla origen: supermercadoetl_prod.silver.precios_productos
- Tablas relacionadas: gold.recetas (consume esta vista)

TRANSFORMACIONES PRINCIPALES:

1. NORMALIZACIÓN DE UNIDADES DE PESO:
   - kg → cantidad_base = 1000 gramos (unidad estándar)
   - medio_kg → cantidad_base = 500 gramos
   - gr → extrae cantidad de descripción usando REGEXP_EXTRACT
   - Ejemplo: "Sal Fina CELUSAL 500g" → cantidad_base = 500

2. NORMALIZACIÓN DE UNIDADES DE VOLUMEN:
   - ltr → cantidad_base = 1000 mililitros (unidad estándar)
   - medio_ltr → cantidad_base = 500 mililitros
   - ml → extrae cantidad de descripción con doble validación:
     * Si contiene "ml": extrae número directo (ej: "2250ml" → 2250)
     * Si contiene "Lt" o "L": convierte a ml (ej: "2,25 Lt" → 2250)
     * Maneja formatos con coma o punto decimal

3. NORMALIZACIÓN DE UNIDADES DISCRETAS:
   - unidad simple → cantidad_base = 1
   - unidad múltiple (sub-unidades) → extrae cantidad del patrón "X Nu"
     * Ejemplo: "Huevo Blanco Maple X 30u" → cantidad_base = 30 huevos
     * Patrón regex: '[Xx] (\\d+)u'
     * Permite calcular precio por huevo individual

4. CÁLCULO DE PRECIO UNITARIO:
   Formula: precio_unitario = ROUND(precio_normalizado / cantidad_base, 4)
   - Precio en $/gramo para productos de peso
   - Precio en $/ml para productos de volumen  
   - Precio en $/unidad para productos discretos

CAMPOS DE SALIDA:
┌──────────────────────┬──────────────┬───────────────────────────────────────┐
│ Campo                │ Tipo         │ Descripción                           │
├──────────────────────┼──────────────┼───────────────────────────────────────┤
│ nombre               │ STRING       │ Identificador del producto            │
│ descripcion          │ STRING       │ Texto completo para referencia        │
│ categoria            │ STRING       │ Clasificación del producto            │
│ precio_normalizado   │ DECIMAL      │ Precio de venta del producto          │
│ unidad               │ STRING       │ Tipo de medida (kg/gr/ltr/ml/unidad)  │
│ cantidad_base        │ INT          │ Cantidad en unidad estándar           │
│ precio_unitario      │ DECIMAL(,4)  │ Precio por unidad base ($/g, $/ml...) │
│ fecha_extraccion     │ DATE         │ Fecha del precio registrado           │
│ url                  │ STRING       │ Trazabilidad al producto origen       │
└──────────────────────┴──────────────┴───────────────────────────────────────┘

CASOS DE USO:
- Cálculo automatizado de costos de recetas
- Comparación de precios entre productos de diferentes presentaciones
- Análisis de precio por unidad de medida estándar
- Base para sistema de planificación de compras

MÉTRICAS CLAVE:
- Productos normalizados: ~200-300 SKUs
- Actualización: Diaria (sincronizada con Silver)
- Granularidad: Producto × Fecha
- Retención: Histórico completo para análisis temporal

REGLAS DE CALIDAD:
✓ cantidad_base > 0 para todos los registros
✓ precio_unitario calculado con 4 decimales de precisión
✓ Validación de NULLIF para evitar divisiones por cero
✓ TRIM aplicado antes de extracciones regex

NOTAS TÉCNICAS:
- Usa CAST(... AS INT) para cantidad_base (valores enteros)
- ROUND(..., 4) para precio_unitario (precisión suficiente)
- ORDER BY categoria facilita consultas por categoría
- Compatible con JOIN a recetas_definicion por campo 'nombre'

MANTENIMIENTO:
- Refresh automático: Al actualizar silver.precios_productos
- Validar nuevos patrones de descripción al agregar categorías
- Monitorear cantidad_base NULL (indica fallo en extracción)

SIGUIENTE PASO EN ARQUITECTURA:
Esta vista alimenta al sistema de cálculo de recetas (gold.recetas) que
realiza matching inteligente y calcula costos totales por receta.

EJEMPLO DE QUERY:
```sql
-- Ver productos normalizados de una categoría
SELECT nombre, precio_normalizado, cantidad_base, precio_unitario
FROM supermercadoetl_prod.gold.catalogo_productos
WHERE categoria = 'Almacén'
  AND fecha_extraccion = CURRENT_DATE()
ORDER BY precio_unitario DESC;
```

CHANGELOG:
- 2026-04-17: Creación inicial con soporte para peso/volumen/unidad
- 2026-04-17: Agregado manejo de sub-unidades (maple de huevos)
- 2026-04-17: Mejorado extracción de ml con conversión de litros

================================================================================
*/

-- ============================================
-- CREACIÓN DE VISTA: catalogo_productos
-- ============================================

CREATE OR REPLACE VIEW supermercadoetl_prod.gold.catalogo_productos AS

-- CTE 1: Extracción y normalización de cantidad base
WITH catalogo_productos AS (
  SELECT 
    nombre,
    descripcion,
    categoria,
    precio_normalizado,
    unidad,
    CAST(CASE 
          -- Unidades completas: kg/ltr → 1000 unidades base
          WHEN unidad IN ('kg','ltr') THEN 1000
          
          -- Medias unidades: medio_kg/medio_ltr → 500 unidades base
          WHEN unidad IN ('medio_ltr','medio_kg') THEN 500
          
          -- CASO ESPECIAL: Productos con múltiples sub-unidades
          -- Ejemplo: "Maple X 30u" → 30 huevos individuales
          WHEN unidad = 'unidad' AND (descripcion LIKE '%X %u%' OR descripcion LIKE '%x %u%') THEN
            CAST(NULLIF(REGEXP_EXTRACT(TRIM(descripcion), '[Xx] (\\d+)u', 1),'') AS INT)
          
          -- Unidad simple: 1 unidad
          WHEN unidad = 'unidad' THEN 1
          
          -- CASO ESPECIAL: Mililitros (doble validación)
          WHEN unidad = 'ml' THEN
            CASE
              -- Patrón directo: "2250ml" → 2250
              WHEN descripcion LIKE '%ml%' THEN 
                CAST(NULLIF(REGEXP_EXTRACT(TRIM(descripcion), '(\\d+)ml', 1),'') AS INT)
              
              -- Patrón con conversión: "2,25 Lt" → 2250 ml
              WHEN descripcion LIKE '%Lt%' OR descripcion LIKE '%L %' OR descripcion LIKE '% L' THEN 
                CAST(CAST(REPLACE(NULLIF(REGEXP_EXTRACT(TRIM(descripcion), '([0-9]+[.,]?[0-9]*)', 1),''), ',', '.') AS DOUBLE) * 1000 AS INT)
              
              ELSE 1
            END
          
          -- Default: Extraer primer número de la descripción
          ELSE CAST(NULLIF(REGEXP_EXTRACT(TRIM(descripcion), '(\\d+)', 0),'') AS INT)
      END AS INT) AS cantidad_base, 
    fecha_extraccion,
    url
  FROM supermercadoetl_prod.silver.precios_productos
),

-- CTE 2: Cálculo de precio unitario normalizado
catalogo_normalizado AS (
  SELECT
    nombre,
    descripcion,
    categoria,
    precio_normalizado,
    unidad,
    cantidad_base,
    ROUND(precio_normalizado / cantidad_base, 4) AS precio_unitario,
    fecha_extraccion,
    url
  FROM catalogo_productos
)

-- Salida final ordenada por categoría
SELECT * FROM catalogo_normalizado
ORDER BY categoria ASC;